In [0]:
   %pip install plotly

In [0]:
from pyspark.sql import functions as F, Window

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
T = "workspace.heatcheck."

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [F.col(c) for c in (lat1, lon1, lat2, lon2)]
    return 6371 * 2 * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(lat2 - lat1) / 2), 2) +
        F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) *
        F.pow(F.sin(F.radians(lon2 - lon1) / 2), 2)))

def rank_scale(df, col, out):
    """Rank each subzone against all others (0.1 = lowest, 1 = highest)."""
    return df.withColumn(out, 0.1 + 0.9 * F.percent_rank().over(Window.orderBy(col)))

wbgt = spark.table(T + "silver_wbgt")

# Heat: average daily peak WBGT per station over the last year
daily = (wbgt.filter(F.col("reading_time") >= F.date_sub(F.current_date(), 365))
    .groupBy("station_id", F.to_date("reading_time").alias("day"))
    .agg(F.max("wbgt").alias("peak_wbgt")))
hazard = daily.groupBy("station_id").agg(
    F.avg("peak_wbgt").alias("avg_peak_wbgt"),
    F.sum(F.when(F.col("peak_wbgt") >= 33, 1).otherwise(0)).alias("high_heat_days"),
    F.count("*").alias("days_observed"))
stations = (wbgt.filter(F.col("latitude").isNotNull())
    .groupBy("station_id")
    .agg(F.first("station_name").alias("station_name"),
         F.avg("latitude").alias("st_lat"), F.avg("longitude").alias("st_lon"))
    .join(hazard, "station_id"))

# Seniors + subzone centre points
subzones = spark.table(T + "silver_subzone_centroids")
seniors = (spark.table(T + "silver_seniors_by_subzone")
    .withColumn("subzone_upper", F.upper(F.trim("subzone")))
    .join(subzones, "subzone_upper")
    .filter(F.col("seniors_65plus") >= 100))

# Nearest WBGT station to each subzone
w = Window.partitionBy("subzone_upper").orderBy("dist_station_km")
nearest_station = (seniors.select("subzone_upper", "sz_lat", "sz_lon").crossJoin(stations)
    .withColumn("dist_station_km", haversine_km("sz_lat", "sz_lon", "st_lat", "st_lon"))
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("subzone_upper", "station_name", "avg_peak_wbgt", "high_heat_days", "dist_station_km"))

# Distance to the nearest eldercare point
elder = spark.table(T + "silver_eldercare_points")
nearest_elder = (seniors.select("subzone_upper", "sz_lat", "sz_lon").crossJoin(elder)
    .withColumn("d", haversine_km("sz_lat", "sz_lon", "lat", "lon"))
    .groupBy("subzone_upper").agg(F.min("d").alias("dist_eldercare_km")))

risk = (seniors.join(nearest_station, "subzone_upper").join(nearest_elder, "subzone_upper")
    .withColumn("share_small_flats", F.col("seniors_small_flats") / F.col("seniors_65plus")))

# Heat x Vulnerable seniors x Distance from help
risk = rank_scale(risk, "seniors_small_flats", "E")     # how many seniors live in 1-2 room flats
risk = rank_scale(risk, "dist_eldercare_km", "V")       # farther from eldercare = less support
hs = risk.agg(F.min("avg_peak_wbgt").alias("mn"), F.max("avg_peak_wbgt").alias("mx")).first()
risk = risk.withColumn("H", 0.7 + 0.3 * (F.col("avg_peak_wbgt") - hs.mn) / ((hs.mx - hs.mn) or 1))
risk = (risk
    .withColumn("risk_raw", F.col("H") * F.col("E") * (0.6 + 0.4 * F.col("V")))
    .withColumn("risk_index", F.round(100 * F.percent_rank().over(Window.orderBy("risk_raw"))).cast("int"))
    .withColumn("risk_band",
        F.when(F.col("risk_index") >= 90, "Check first")
         .when(F.col("risk_index") >= 70, "Check next")
         .otherwise("Monitor"))
    .withColumn("risk_rank", F.row_number().over(Window.orderBy(F.desc("risk_raw")))))

(risk.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T + "gold_subzone_risk"))
print("Subzones scored:", risk.count())
print(risk.groupBy("risk_band").count().orderBy("risk_band").collect())
display(risk.select("risk_rank", "risk_band", "planning_area", "subzone", "risk_index",
                    "seniors_65plus", "seniors_small_flats",
                    F.round("share_small_flats", 2).alias("share_small"),
                    F.round("avg_peak_wbgt", 1).alias("avg_peak_wbgt"),
                    F.round("dist_eldercare_km", 2).alias("km_to_eldercare"))
        .orderBy("risk_rank").limit(15))

In [0]:
import os, json, re
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Patch
from matplotlib.collections import PatchCollection

T = "workspace.heatcheck."

def kml_attrs(desc):
    return dict(re.findall(r"<th>(.*?)</th>\s*<td>(.*?)</td>", desc or ""))

geo = json.load(open("/Volumes/workspace/heatcheck/raw/reference/subzones.geojson"))
for feat in geo["features"]:
    if "SUBZONE_N" not in feat["properties"]:
        feat["properties"] = kml_attrs(feat["properties"].get("Description"))

pdf = spark.table(T + "gold_subzone_risk").select(
    "subzone_upper", "subzone", "planning_area", "risk_rank", "risk_band",
    "seniors_small_flats", "sz_lat", "sz_lon").toPandas()
band = dict(zip(pdf["subzone_upper"], pdf["risk_band"]))
n_first = int((pdf["risk_band"] == "Check first").sum())

COLOURS = {"Check first": "#c62828", "Check next": "#f4a261", "Monitor": "#d9d9d9"}
NO_DATA = "#f2f2f2"

fig = plt.figure(figsize=(16, 9), dpi=200)
fig.patch.set_facecolor("white")
ax = fig.add_axes([0.01, 0.06, 0.63, 0.78])

# Subzone shapes, coloured by action band
patches, colours = [], []
for feat in geo["features"]:
    geom = feat["geometry"]
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    for poly in polys:
        patches.append(Polygon([(p[0], p[1]) for p in poly[0]], closed=True))
        colours.append(COLOURS.get(band.get(feat["properties"]["SUBZONE_N"]), NO_DATA))
ax.add_collection(PatchCollection(patches, facecolor=colours, edgecolor="white", linewidth=0.5))

# Numbered markers for the top 5
top5 = pdf.nsmallest(5, "risk_rank")
for r in top5.itertuples():
    ax.scatter(r.sz_lon, r.sz_lat, s=320, color="#5a0000", edgecolor="white", linewidth=1.5, zorder=5)
    ax.text(r.sz_lon, r.sz_lat, str(r.risk_rank), color="white", ha="center", va="center",
            fontsize=10, fontweight="bold", zorder=6)

ax.set_xlim(103.6, 104.09)
ax.set_ylim(1.2, 1.48)
ax.set_aspect("equal")
ax.axis("off")

legend = [Patch(facecolor=COLOURS[k], label=k) for k in COLOURS] + \
         [Patch(facecolor=NO_DATA, edgecolor="#cccccc", label="Not scored (under 100 seniors)")]
ax.legend(handles=legend, loc="lower right", frameon=False, fontsize=11)

# Title, subtitle and the top 5 list
fig.text(0.02, 0.93, f"{n_first} subzones where AAC staff should check on seniors first",
         fontsize=22, fontweight="bold", color="#1a1a1a")
fig.text(0.02, 0.885, "Risk = heat  ×  seniors in HDB 1-2 room flats  ×  distance from eldercare",
         fontsize=13, color="#555555")

fig.text(0.67, 0.78, "Top 5 subzones", fontsize=16, fontweight="bold", color="#1a1a1a")
for i, r in enumerate(top5.itertuples()):
    fig.text(0.67, 0.72 - i * 0.075, f"{r.risk_rank}.  {r.subzone}", fontsize=14, fontweight="bold", color="#5a0000")
    fig.text(0.69, 0.692 - i * 0.075,
             f"{r.planning_area}  ·  {int(r.seniors_small_flats):,} seniors in 1-2 room flats",
             fontsize=11, color="#444444")

fig.text(0.02, 0.02,
         "Data: NEA WBGT readings (last 12 months), SingStat residents by subzone (June 2026), "
         "MOH eldercare services, URA Master Plan 2019 subzones.",
         fontsize=9, color="#888888")

out_dir = "/Volumes/workspace/heatcheck/raw/exports"
os.makedirs(out_dir, exist_ok=True)
fig.savefig(f"{out_dir}/heatcheck_map_v0.png", dpi=200, facecolor="white")
plt.show()

In [0]:
import os, re, json, copy
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

T = "workspace.heatcheck."
OUT_DIR = "/Volumes/workspace/heatcheck/raw/exports"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Helpers ----------
def kml_attrs(desc):
    return dict(re.findall(r"<th>(.*?)</th>\s*<td>(.*?)</td>", desc or ""))

def ring_area(ring):
    return sum(x1 * y2 - x2 * y1 for (x1, y1, *_), (x2, y2, *_) in zip(ring, ring[1:]))

def rewind(geojson):
    """This map type needs outer edges drawn clockwise. Fix any that aren't."""
    g = copy.deepcopy(geojson)
    for feat in g["features"]:
        geom = feat["geometry"]
        polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        for poly in polys:
            for i, ring in enumerate(poly):
                clockwise = ring_area(ring) < 0
                if (i == 0 and not clockwise) or (i > 0 and clockwise):
                    ring.reverse()
    return g

def heat_level(w):
    return "High" if w >= 33 else "Moderate" if w >= 31 else "Low"

# ---------- Data ----------
geo = json.load(open("/Volumes/workspace/heatcheck/raw/reference/subzones.geojson"))
for feat in geo["features"]:
    if "SUBZONE_N" not in feat["properties"]:
        feat["properties"] = kml_attrs(feat["properties"].get("Description"))
geo_fixed = rewind(geo)

risk = spark.table(T + "gold_subzone_risk").select(
    "subzone_upper", "subzone", "planning_area", "risk_rank", "risk_band",
    "seniors_65plus", "seniors_small_flats", "share_small_flats", "avg_peak_wbgt",
    "high_heat_days", "dist_eldercare_km", "station_name", "dist_station_km",
    "sz_lat", "sz_lon").toPandas()
n_scored = len(risk)
n_first = int((risk["risk_band"] == "Check first").sum())

def hover_text(r):
    if r["risk_band"] == "Not scored":
        return f"<b>{r['subzone']}</b> ({r['planning_area']})<br>Not scored: fewer than 100 seniors live here."
    return (
        f"<b>{r['subzone']}</b> ({r['planning_area']})<br>"
        f"<b>{r['risk_band']}</b>: risk rank {int(r['risk_rank'])} of {n_scored} subzones<br><br>"
        f"Seniors aged 65+: {int(r['seniors_65plus']):,}<br>"
        f"Seniors in HDB 1-2 room flats: {int(r['seniors_small_flats']):,} ({r['share_small_flats']:.0%} of seniors)<br>"
        f"Typical daily peak heat stress: {r['avg_peak_wbgt']:.1f}°C WBGT ({heat_level(r['avg_peak_wbgt'])})<br>"
        f"Days with High heat stress in the past year: {int(r['high_heat_days'])}<br>"
        f"Nearest eldercare centre: {r['dist_eldercare_km']:.1f} km away<br>"
        f"Heat readings from: {r['station_name']} station ({r['dist_station_km']:.1f} km away)")

# Every subzone on the map, so unscored areas still appear
all_sz = pd.DataFrame([{"subzone_upper": f["properties"]["SUBZONE_N"],
                        "pa_map": f["properties"].get("PLN_AREA_N", "")} for f in geo["features"]])
full = all_sz.merge(risk, on="subzone_upper", how="left")
full["risk_band"] = full["risk_band"].fillna("Not scored")
full["subzone"] = full["subzone"].fillna(full["subzone_upper"].str.title())
full["planning_area"] = full["planning_area"].fillna(full["pa_map"].str.title())

LABELS = {"Check first": "Check first (highest 10% risk)",
          "Check next": "Check next (next 20%)",
          "Monitor": "Monitor (remaining 70%)",
          "Not scored": "Not scored (under 100 seniors)"}
COLOURS = {LABELS["Check first"]: "#c62828", LABELS["Check next"]: "#f4a261",
           LABELS["Monitor"]: "#d9d9d9", LABELS["Not scored"]: "#f2f2f2"}
full["band_label"] = full["risk_band"].map(LABELS)
full["hover"] = full.apply(hover_text, axis=1)

# ---------- Map ----------
fig = px.choropleth(
    full, geojson=geo_fixed, locations="subzone_upper", featureidkey="properties.SUBZONE_N",
    color="band_label", color_discrete_map=COLOURS,
    category_orders={"band_label": list(LABELS.values())}, custom_data=["hover"])
fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>",
                  marker_line_color="white", marker_line_width=0.5,
                  selector=dict(type="choropleth"))

top5 = risk.nsmallest(5, "risk_rank").copy()
top5["hover"] = top5.apply(hover_text, axis=1)
fig.add_trace(go.Scattergeo(
    lat=top5["sz_lat"], lon=top5["sz_lon"], mode="markers+text",
    marker=dict(size=18, color="#5a0000", line=dict(color="white", width=1.5)),
    text=top5["risk_rank"].astype(str), textposition="middle center",
    textfont=dict(color="white", size=11),
    customdata=top5[["hover"]].values, hovertemplate="%{customdata[0]}<extra></extra>",
    showlegend=False))

fig.update_geos(fitbounds="locations", visible=False, bgcolor="white",
                domain=dict(x=[0, 0.66], y=[0, 0.95]))

# ---------- Information panel ----------
top5_lines = "<br>".join(f"{int(r.risk_rank)}. {r.subzone} ({r.planning_area})" for r in top5.itertuples())
panel = (
    "<b>How to read this map</b><br>"
    "Each area is a URA subzone, coloured by how much support<br>"
    "its seniors may need on very hot days.<br><br>"
    "<span style='color:#c62828'>■</span> <b>Check first</b>: the highest-risk 10% of subzones.<br>"
    "<i>Call or visit seniors here before and during hot spells.</i><br>"
    "<span style='color:#f4a261'>■</span> <b>Check next</b>: the next 20%.<br>"
    "<i>Add to outreach on days with High heat stress.</i><br>"
    "<span style='color:#bdbdbd'>■</span> <b>Monitor</b>: the remaining 70%.<br>"
    "<i>Routine check-ins are usually enough.</i><br><br>"
    "<b>What the risk score combines</b><br>"
    "1. Heat: typical daily peak heat stress nearby (NEA WBGT)<br>"
    "2. Seniors aged 65+ living in HDB 1-2 room flats<br>"
    "3. Distance to the nearest eldercare centre<br><br>"
    "<b>NEA heat stress levels (WBGT)</b><br>"
    "Low: below 31°C  |  Moderate: 31 to 33°C  |  High: 33°C and above<br><br>"
    f"<b>Top 5 subzones</b><br>{top5_lines}<br><br>"
    "<i>Hover over any area for details. Scroll to zoom,<br>"
    "drag to move, double-click to reset.</i><br><br>"
    "<span style='font-size:11px;color:#777777'>"
    "Data: NEA WBGT readings (last 12 months), SingStat residents<br>"
    "by subzone (June 2026), MOH eldercare services (2024), URA<br>"
    "Master Plan 2019 subzones. HeatCheck v0 prototype, not yet<br>"
    "validated for operational use.</span>")
fig.add_annotation(x=0.68, y=0.93, xref="paper", yref="paper", xanchor="left", yanchor="top",
                   align="left", showarrow=False, text=panel, bgcolor="#fafafa",
                   bordercolor="#dddddd", borderwidth=1, borderpad=12, font=dict(size=13))

fig.update_layout(
    title=dict(text=("<b>HeatCheck: where seniors are most at risk from heat</b><br>"
                     f"<sup>{n_first} subzones to check first  |  "
                     "Risk = heat × seniors in HDB 1-2 room flats × distance from eldercare</sup>"),
               x=0.01, y=0.97),
    font=dict(family="Arial, Helvetica, sans-serif", size=13, color="#222222"),
    height=860, margin=dict(l=10, r=10, t=90, b=10),
    paper_bgcolor="white", template="plotly_white",
    legend=dict(title=dict(text="<b>Suggested action</b>"), x=0.01, y=0.9,
                bgcolor="rgba(255,255,255,0.92)", bordercolor="#dddddd", borderwidth=1),
    hoverlabel=dict(bgcolor="white", font_size=13, align="left"))

fig.write_html(f"{OUT_DIR}/heatcheck_map_interactive.html", include_plotlyjs=True,
               config={"scrollZoom": True, "displaylogo": False})
print("Saved interactive map")